# Corpus additions v2 ? supervised CPU script
No GPU or leaderboard. Each poll cell stays below ten minutes; total runtime capped at eight hours.

In [ ]:
MODULES = {'content_parser.py': '"""Extract source content from the host templates observed in the pilot.\n\nNo network calls. Unknown or incomplete pages stay in the review queue.\n"""\nimport argparse\nimport base64\nimport binascii\nimport gzip\nimport json\nimport re\nfrom collections import Counter\nfrom pathlib import Path\nfrom urllib.parse import parse_qs, urljoin, urlsplit, parse_qsl\nfrom html import unescape\n\nfrom bs4 import BeautifulSoup, NavigableString, Tag\n\nVERSION = \'parser-v2.5\'\n\n# Public CMS renderer: dSecurity.b7 in dcore-2.2.0.d7.204.js and\n# dcms.decode in cmsfcore.dg72.js (baophutho.vn / baothanhhoa.vn).\nCMS_ALPHABET = \' 0123456789aáàạảãăắằặẳẵâấầậẩẫbcdđeéẹẻẽêếềệểễfghiíìịỉĩjklmnoóòọỏõôốồộổỗơớờợởỡpqrstuúùụủũưứừựửữvwxyýỳỵỷỹAÁÀẠẢÃĂẮẰẶẲẴÂẤẦẬẨẪBCDĐEÉẸẺẼÊẾỀỆỂỄFGHIÍÌỊỈĨJKLMNOÓÒỌỎÕÔỐỒỘỔỖƠỚỜỢỞỠPQRSTUÚÙỤỦŨƯỨỪỰỬỮVWXYÝỲỴỶỸ@#=$%*&?:;,.!()<>{}[]"\'\n\n\ndef decode_cms(soup, result):\n    """Reproduce the site\'s public renderer; never execute source JavaScript."""\n    if result[\'host\'] not in {\'baophutho.vn\', \'baothanhhoa.vn\'}:\n        return\n    node = soup.select_one(\'#dcontent\')\n    if node is None or node.find():\n        return\n    payload = re.sub(r\'\\s+\', \'\', node.get_text())\n    scripts = \'\\n\'.join(n.get_text() for n in soup.select(\'script:not([src])\'))\n    key = re.search(r\'\'\'dmain\\.detailView\\(\\s*[\'"][^\'"]*[\'"]\\s*,\\s*[\'"](\\d+)[\'"]\'\'\', scripts)\n    if not key or not payload:\n        return\n    try:\n        encoded = base64.b64decode(payload, validate=True).decode(\'utf-8\')\n        shift = int(key[1])\n        decoded = encoded.translate(str.maketrans(CMS_ALPHABET, \'\'.join(\n            CMS_ALPHABET[(i - shift) % len(CMS_ALPHABET)] for i in range(len(CMS_ALPHABET)))))\n        fragment = BeautifulSoup(decoded, \'html.parser\')\n        # Accept only the known article structure. Failed/new encodings stay queued.\n        if not fragment.select_one(\'.pBody, .pHead, .pTitle\') or \'\\ufffd\' in decoded:\n            return\n    except (ValueError, UnicodeError, binascii.Error):\n        return\n    result[\'normalizations\'].append(dict(reason=\'cms_base64_shift\', source_path=\'#dcontent\',\n                                        key=shift, text=payload))\n    node.clear()\n    node.extend(list(fragment.contents))\n\n\ndef longchau_regions(soup, result):\n    """Read explicit article/disease fields, including sections hidden by \'read more\'."""\n    script = soup.select_one(\'script#__NEXT_DATA__\')\n    try:\n        props = json.loads(script.get_text())[\'props\'][\'pageProps\']\n        disease = urlsplit(result.get(\'final_url\') or result[\'url\']).path.startswith(\'/benh/\')\n        data = props[\'data\'] if disease else props[\'detail\'][\'data\']\n        prefix = \'props.pageProps.data\' if disease else \'props.pageProps.detail.data\'\n        if data[\'slug\'].strip(\'/\') != urlsplit(result.get(\'final_url\') or result[\'url\']).path.strip(\'/\'):\n            result[\'warnings\'].append(\'structured_source_url_mismatch\')\n            return []\n    except (AttributeError, KeyError, TypeError, ValueError):\n        return []\n    result[\'title\'] = data.get(\'headline\') or data.get(\'name\') or result[\'title\']\n    result[\'page_type\'] = \'encyclopedia\' if disease else \'article\'\n    result[\'source_metadata\'] = {k: data.get(k) for k in (\'author\', \'approver\', \'publishedAt\', \'updatedAt\')}\n    regions = []\n    def add(field, value, role=\'body\', heading=None):\n        if not value:\n            return\n        if not isinstance(value, str):\n            result[\'warnings\'].append(\'structured_field_not_parsed\')\n            result[\'missing_sections\'].append(field)\n            return\n        root = soup.new_tag(\'section\')\n        root[\'_source_path\'] = f\'script#__NEXT_DATA__::{prefix}.{field}\'\n        if heading:\n            h = soup.new_tag(\'h2\')\n            h.string = heading\n            root.append(h)\n        root.extend(list(BeautifulSoup(value, \'html.parser\').contents))\n        regions.append((root, role))\n        result[\'selectors\'].append(root[\'_source_path\'])\n    add(\'shortDescription\', data.get(\'shortDescription\'), \'summary\')\n    add(\'description\', data.get(\'description\'))\n    for field, heading in [(\'symptom\', \'Triệu chứng\'), (\'aetiologies\', \'Nguyên nhân\'),\n                           (\'risk\', \'Nguy cơ\'), (\'diagnoseAndTreaty\', \'Chẩn đoán và điều trị\'),\n                           (\'livingAndPreventive\', \'Sinh hoạt và phòng ngừa\')]:\n        add(field, data.get(field), heading=heading)\n    for index, faq in enumerate(data.get(\'faqs\') or []):\n        if isinstance(faq, dict):\n            add(f\'faqs[{index}].answer\', faq.get(\'answer\'), \'faq\', faq.get(\'question\'))\n    add(\'referenceSource\', data.get(\'referenceSource\'), \'references\')\n    add(\'sourceArticle\', data.get(\'sourceArticle\'), \'references\')\n    for kind, value in [(\'audio\', data.get(\'audio\')), (\'video\', data.get(\'urlVideo\'))]:\n        url = value.get(\'url\') if isinstance(value, dict) else value\n        if isinstance(url, str) and url.startswith((\'http://\', \'https://\')):\n            result[\'resources\'].append(dict(kind=kind, url=url, caption=\'\'))\n    if any((props.get(\'mediaHub\') or {}).values()):\n        result[\'structured_media\'] = props[\'mediaHub\']\n        result[\'warnings\'].append(\'embedded_content_not_parsed\')\n    return regions\n\n# Selectors are ordered fallbacks, except for the explicit multi-region cases below.\nRULES = {}\n\ndef register(hosts, selectors):\n    for host in hosts.split():\n        RULES[host] = list(selectors)\n\nregister(\'suckhoecongdongonline.vn nhandan.vn baogialai.com.vn tienphong.vn baoquangninh.vn www.sggp.org.vn www.vietnamplus.vn\', [\'.article__body\'])\nregister(\'suckhoedoisong.vn thanhnien.vn tuoitre.vn baochinhphu.vn\', [\'.detail-content\'])\nregister(\'baohaiphong.vn baodanang.vn hanoimoi.vn\', [\'.entry\'])\nregister(\'baonghean.vn khoahocphothong.vn\', [\'article.entry\'])\nregister(\'phuyen.baodaklak.vn baoquangtri.vn baovinhlong.com.vn\', [\'.td-post-content\'])\nregister(\'baolangson.vn baotayninh.vn\', [\'#content-detail\'])\nregister(\'benhviennhitrunguong.gov.vn special.vietnamplus.vn\', [\'.entry-content\'])\nregister(\'baby.39.net care.39.net fitness.39.net woman.39.net food.39.net fk.39.net cancer.39.net nk.39.net heart.39.net man.39.net gan.39.net tnb.39.net wei.39.net shen.39.net ek.39.net gk.39.net gc.39.net pf.39.net sj.39.net byby.39.net\', [\'.art_content\', \'#contentText.art_con\'])\nregister(\'www.a-hospital.com zhongyibaodian.net\', [\'#bodyContent\'])\nRULES.update({\n    \'www.cnkang.com\': [\'#endText\', \'.detailc\', \'#art_text\', \'.conmain > div:not([class])\', \'td.style3:has(> p)\', \'.detail:has(.ysxp02c_table)\', \'.zh05b\'],\n    \'www.zydcd.com\': [\'div.info_txt\', \'#booklist\'],\n    \'www.wujue.com\': [\'.art-text\', \'td#text\', \'#article-main_left #art-leftbox\', \'td:has(img.photoresize)\'],\n    \'www.youlai.cn\': [\'[class*="textContent--"]\'],\n    \'phunusuckhoe.giadinhonline.vn\': [\'#article-body\'],\n    \'www.vinmec.com\': [\'#main-article\'],\n    \'medlatec.vn\': [\'.block-posts-single .description\', \'.infrastructure-description\'],\n    \'qihuangzhishu.com\': [\'.cbody\'],\n    \'giadinhonline.vn\': [\'#explus-editor\'],\n    \'vietnamnet.vn\': [\'#maincontent\', \'.maincontent.content-full-image\', \'.emag-inner .sapo, .emag-inner .titles, .emag-inner .inner-contetn\'],\n    \'test.pmphai.com\': [\'#content.gu_det_left_con\'],\n    \'tiemchunglongchau.com.vn\': [\'#content-article\', \'section[aria-label="Quiz Details Body"]\'],\n    \'baocantho.com.vn\': [\'#newscontents .content-fck\'],\n    \'hellobacsi.com\': [\'.article-content-item.body-content\'],\n    \'baoangiang.com.vn\': [\'#newscontents .content-fck\'],\n    \'vov.vn\': [\'.text-long\', \'.article-content\', \'.node-content-audio .node-body > .content\', \'article.detail--video > .content\', \'.box-content .article-media\'],\n    \'jb39.com\': [\'.post-jibing\'],\n    \'youmed.vn\': [\'section.article-content article:not([class]) > div.prose\'],\n    \'vov2.vov.vn\': [\'.article-body\', \'.article__body\'],\n    \'www.msdmanuals.cn\': [\'[class*="TopicMainContent_content"]\', \'[class*="Chapter_content"]\', \'[class*="Section_content"]\'],\n    \'suckhoeviet.org.vn\': [\'.__MASTERCMS_CONTENT\'],\n    \'tamanhhospital.vn\': [\'.post_info.box_detail_post:has(p)\'],\n    \'www.qdnd.vn\': [\'.post-content\'],\n    \'thaythuocvietnam.vn\': [\'.content-post.clearfix\'],\n    \'bachmai.gov.vn\': [\'.content-editor-post\'],\n    \'www.baidu.com\': [\'.health-dict__overview\'],\n    \'dantri.com.vn\': [\'#desktop-in-article\', \'#desktop-cate-in-article\', \'.e-magazine__body\', \'.video-container\'],\n    \'vnexpress.net\': [\'article.fck_detail\'],\n    \'health.people.com.cn\': [\'#rm_txt_zw\', \'.artDet\', \'.articleCont\', \'.rm_txt_con\'],\n    \'baophutho.vn\': [\'#dcontent\'],\n    \'baothanhhoa.vn\': [\'#dcontent\', \'.article__body\'],\n    \'www.pharmacity.vn\': [\'#pmc-content-blog-detail\'],\n    \'www.familydoctor.cn\': [\'.article-content\'],\n    \'benhvienvietduc.org\': [\'.news-primary-content .edittor-content\', \'.news-primary-content\'],\n    \'mega.vietnamplus.vn\': [\'.entry.production.longform\'],\n    \'fk.99.com.cn\': [\'.wrap-left .content\'],\n})\nRULES[\'nhandan.vn\'].append(\'.Theme-Layer-BodyText--inner\')\nRULES[\'care.39.net\'].append(\'.art > .content\')\nRULES[\'man.39.net\'].append(\'body > p\')\n# Older 39.net article templates, shared across channel subdomains.\nfor host in [h for h in RULES if h.endswith(\'.39.net\')]:\n    RULES[host] += [s for s in (\'.art > .content\', \'.artbox > .article\', \'body > .wrap:not(.tools):has(> p)\')\n                    if s not in RULES[host]]\nRULES[\'www.cnkang.com\'].append(\'td#iContent\')\nRULES[\'suckhoedoisong.vn\'] += [\'.detail-container [data-role="video-autoplay"]\',\n                               # e-magazine layout and video articles whose body is only the player.\n                               \'.sp-detail-content\', \'.detail-frame-video\']\nRULES[\'vov2.vov.vn\'].append(\'article.node--emagazine .content.node-emagazine\')\nRULES[\'dantri.com.vn\'].append(\'main > .video-highlight, main > .video-grid\')\n\nSUMMARY_SELECTORS = \'.article__sapo, .detail-sapo, .detail__sapo, .post-summary, .sc-longform-header-sapo, .content-detail-sapo, .article-summary, .art_summary\'\nSUMMARY_HOST = {\n    \'giadinhonline.vn\': \'h2.intro\', \'medlatec.vn\': \'.block-posts-single .shortdescription\',\n    \'dantri.com.vn\': \'[data-slot="sapo"]\', \'fk.99.com.cn\': \'.introduct\',\n    \'www.familydoctor.cn\': \'article.article .content > .summary\',\n    \'vov2.vov.vn\': \'.article-description\',\n}\n\n# Only identified UI containers; ordinary links, lists, tables and references stay.\nCOMMON_REMOVE = \'script, style, button, form, .ez-toc-container, #ez-toc-container, #toc, #toc-container, .sharethis-inline-share-buttons, .bdsharebuttonbox, .related-news, article.story, [type="RelatedNewsBox"], [type="RelatedOneNews"], .readmore-body-box, .article__meta, .article__source, .article__tag\'\nHOST_REMOVE = {\n    \'benhvienvietduc.org\': \'.news-details-date, .news-detail-bot, .news-detail-more, .news-primary-content > h1\',\n    \'www.a-hospital.com\': \'#jump-to-nav, table.nav, table.navbox, table.msg-table, #catlinks, #fromlink, .editsection\',\n    \'zhongyibaodian.net\': \'#jump-to-nav, table.nav, table.navbox, table.msg-table, #catlinks, #fromlink, .editsection\',\n    \'www.cnkang.com\': \'#jkpl, .detail > h1, .detail > .detailb\',\n    \'jb39.com\': \'.jb-daohang, .g160, .zz-more, .post-bm-ad, .r2, .jb-xx-jb, [class*="jb-search-"], ul.ul-ss-2.h2-ss, .jb-page, .jb-dir-page\',\n    \'thaythuocvietnam.vn\': \'.post-views\',\n    \'qihuangzhishu.com\': \'.book-page, .book-page2, .fhbsml\',\n    \'fk.99.com.cn\': \'ul.cont-text, .cont-tag\',\n    \'baothanhhoa.vn\': \'.drelated\',\n    \'vietnamnet.vn\': \'.inner-article\',\n    \'tamanhhospital.vn\': \'.ggnews_wrap, .div_over.bg_xam\',\n    \'www.baidu.com\': \'.health-dict__overview__navs, .health-dict__overview__extend, .health-dict__image-viewer__wrapper\',\n    \'special.vietnamplus.vn\': \'.sharedaddy, #jp-relatedposts, .wp-block-buttons\',\n}\nBLOCK_TAGS = {\'p\', \'div\', \'section\', \'article\', \'li\', \'ul\', \'ol\', \'blockquote\', \'figure\', \'figcaption\', \'dl\', \'dt\', \'dd\', \'tr\', \'td\', \'th\', \'thead\', \'tbody\', \'tfoot\', \'caption\', \'table\'}\n\ndef text(node):\n    return re.sub(r\'\\s+\', \' \', node.get_text(\' \', strip=True) if isinstance(node, Tag) else str(node)).replace(\'\\x00\', \'\').strip()\n\ndef decode(raw, content_type=\'\'):\n    probe = raw[:8192].decode(\'ascii\', errors=\'ignore\')\n    charset = re.search(r\'charset\\s*=\\s*["\\x27]?([\\w-]+)\', probe, re.I)\n    header = re.search(r\'charset=([\\w-]+)\', content_type, re.I)\n    preferred = charset.group(1) if charset else header.group(1) if header else \'utf-8\'\n    if preferred.lower() in {\'gbk\', \'gb2312\', \'gb-2312\'}:\n        preferred = \'gb18030\'\n    for encoding in dict.fromkeys([preferred, \'utf-8\', \'gb18030\']):\n        try:\n            return raw.decode(encoding), encoding, [] if encoding == preferred else [\'encoding_fallback\']\n        except (UnicodeDecodeError, LookupError):\n            pass\n    return raw.decode(\'utf-8\', errors=\'replace\'), \'utf-8\', [\'encoding_replacement\']\n\ndef top_nodes(nodes):\n    chosen = {id(n) for n in nodes}\n    return [n for n in nodes if not any(id(p) in chosen for p in n.parents)]\n\ndef title_of(soup, host):\n    selectors = {\n        \'www.120ask.com\': \'#d_askH1\', \'ask.39.net\': \'.ask_tit\',\n        \'www.a-hospital.com\': \'#firstHeading\', \'zhongyibaodian.net\': \'#firstHeading\',\n        \'test.pmphai.com\': \'.gu_det_left_top\',\n        \'www.baidu.com\': \'.health-dict__disease__header__info__title\',\n    }\n    if host in selectors:\n        for n in soup.select(selectors[host]):\n            if text(n):\n                return text(n)\n    for selector in [\'meta[property="og:title"]\', \'meta[name="twitter:title"]\']:\n        for n in soup.select(selector):\n            if n.get(\'content\', \'\').strip():\n                return n[\'content\'].strip()\n    for selector in (\'h1\', \'title\'):\n        for n in soup.select(selector):\n            if text(n):\n                return text(n)\n    return \'\'\n\n\ndef normalize_markup(root, result):\n    """Decode escaped formatting inside content, retaining the original text for review."""\n    pattern = r\'</?(?:p|div|br|img|a|strong|b|em|i|span|h[1-6]|ul|ol|li|table|tr|td|th)(?:\\s[^<>]*|\\s*/?)>\'\n    for string in list(root.find_all(string=True)):\n        if type(string) is not NavigableString:\n            continue\n        value = str(string)\n        decoded = unescape(value)\n        if not re.search(pattern, decoded, re.I):\n            continue\n        fragment = BeautifulSoup(decoded, \'html.parser\')\n        if not fragment.find():\n            continue\n        result[\'normalizations\'].append(dict(reason=\'escaped_html\',\n                                            source_path=dom_path(string.parent), text=value))\n        result[\'warnings\'].append(\'escaped_html_normalized\')\n        string.replace_with(*list(fragment.contents))\n\n\ndef find_pagination(soup, regions, url):\n    candidates = soup.select(\'a[rel~="next"], .pagination a[href], .pages a[href], #pages a[href]\')\n    candidates += [link for root, _ in regions for link in root.select(\'a[href]\')]\n    source = urlsplit(url)\n    def article_key(parts):\n        path = re.sub(r\'_\\d+(?=\\.(?:s?html?|htm)$)\', \'\', parts.path, flags=re.I)\n        query = sorted((key, value) for key, value in parse_qsl(parts.query)\n                       if key.lower() not in {\'page\', \'p\', \'pageno\', \'pagenum\', \'pageindex\', \'start\', \'offset\'})\n        return path, query\n    links = {}\n    for link in candidates:\n        if \'next\' not in link.get(\'rel\', []) and not re.fullmatch(\n                r\'下一页|下页|下一頁|下一张|下一張|Trang tiếp(?: theo)?|Next(?: page)?|[>»›]+\', text(link), re.I):\n            continue\n        target = urlsplit(urljoin(url, link.get(\'href\', \'\')))\n        absolute = target._replace(fragment=\'\').geturl()\n        if (target.scheme not in {\'http\', \'https\'} or target.hostname != source.hostname\n                or absolute == source._replace(fragment=\'\').geturl()):\n            continue\n        same_article = article_key(target) == article_key(source)\n        # Book chapters and disease detail tabs have separate URLs. Their own\n        # missing-section warnings already describe these links.\n        if source.hostname in {\'qihuangzhishu.com\', \'jb39.com\'} and not same_article:\n            continue\n        links[absolute] = dict(url=absolute, same_article=same_article)\n    return list(links.values())\n\ndef dom_path(node):\n    if isinstance(node, Tag) and node.get(\'_source_path\'):\n        return node[\'_source_path\']\n    parts = []\n    while isinstance(node, Tag) and node.name != \'[document]\':\n        if node.get(\'_source_path\'):\n            parts.append(node[\'_source_path\'])\n            break\n        part = node.name\n        if node.get(\'id\'):\n            parts.append(part + \'#\' + node[\'id\'])\n            break\n        index = 1 + sum(1 for s in node.previous_siblings if isinstance(s, Tag) and s.name == node.name)\n        parts.append(f\'{part}:nth-of-type({index})\')\n        node = node.parent\n    return \' > \'.join(reversed(parts))\n\ndef blocks_from(root, role):\n    """Walk every text node once. Nested lists/tables are not flattened twice."""\n    blocks = []\n    def emit(kind, value, node):\n        value = text(value)\n        if value:\n            blocks.append({\'kind\': kind, \'role\': role, \'text\': value, \'source_path\': dom_path(node)})\n    def walk(node, kind=\'paragraph\'):\n        if node.name == \'tr\':\n            cells = [cell for cell in node.find_all([\'td\', \'th\']) if cell.find_parent(\'tr\') is node]\n            # html.parser can nest unclosed cells/rows. Walk that DOM once instead\n            # of flattening an ancestor row and then emitting its descendants again.\n            if cells and not any(c.find(\'table\') for c in cells):\n                values = [\' \'.join(str(string) for string in cell.find_all(string=True)\n                                   if type(string) is NavigableString\n                                   and string.find_parent([\'td\', \'th\']) is cell) for cell in cells]\n                emit(\'table_row\', \' | \'.join(values), node)\n                for row in node.find_all(\'tr\'):\n                    if row.find_parent(\'tr\') is node:\n                        walk(row)\n                return\n        pending = []\n        def flush():\n            emit(kind, \'\'.join(pending), node)\n            pending.clear()\n        for child in node.children:\n            if isinstance(child, NavigableString):\n                # Comments are not document text.\n                if child.__class__ is NavigableString:\n                    pending.append(str(child))\n            elif isinstance(child, Tag):\n                if child.name in {\'img\', \'source\', \'iframe\'}:\n                    continue\n                if child.name == \'br\':\n                    flush()\n                elif child.name == \'table\':\n                    flush()\n                    walk(child)\n                elif re.fullmatch(r\'h[1-6]\', child.name) or set(child.get(\'class\', [])) & {\'gu_det_left_con_h1\', \'pInterTitle\', \'health-dict__overview__text__level1-tag__title\', \'disease_title\'}:\n                    flush()\n                    emit(\'heading\', text(child), child)\n                elif child.name in BLOCK_TAGS:\n                    flush()\n                    walk(child, \'list_item\' if child.name == \'li\' else \'caption\' if child.name in {\'figcaption\', \'caption\'} else \'paragraph\')\n                else:\n                    # Inline tags remain in their original position; no artificial\n                    # spaces are inserted inside Chinese words split by links.\n                    pending.append(child.get_text(\'\', strip=False))\n        flush()\n    walk(root)\n    return blocks\n\n\nWIKI_REFERENCE_HEADINGS = {\'参考\', \'参考文献\', \'参考来源\', \'参考资料\', \'注释\', \'引用\'}\n# Site-wide templates appended to a-hospital/zhongyibaodian entries (sample of 1,353 pages).\n# Link density alone is not enough: clinical lists such as 临床意义 link every disease name.\nWIKI_NAVIGATION_HEADINGS = re.compile(\n    r\'参看|参见|另见|相关页面|相关条目|更多医学百科条目|站外链接|外部链接|外部连结|人体穴位图|乙肝科普知识\'\n    r\'|附：中药材大全|按功效分类的常用方剂|.*专题|健康问答网关于.+的相关提问|.*按.+查询.*|.+分区域查询\')\n\ndef mostly_links(nodes):\n    total = sum(len(re.sub(r\'\\s+\', \'\', text(n))) for n in nodes)\n    linked = sum(len(re.sub(r\'\\s+\', \'\', text(a))) for n in nodes\n                 for a in ([n] if n.name == \'a\' else n.select(\'a\')))\n    return total > 0 and linked >= total / 2\n\n\ndef trim_wiki_tail(root, remove):\n    """Drop trailing MediaWiki navigation sections and empty headings.\n\n    Sections are scanned from the end. Reference and media-only sections are kept\n    and scanning continues; the first other section stops trimming. Returns False\n    when the remaining tail still needs review: its last section (or last block,\n    without headings) is mostly links under an unrecognized heading. Returns\n    \'listing\' without removing anything when every section is navigation, i.e.\n    the page itself is a lookup index."""\n    children = root.find_all(recursive=False)\n    sections, current = [], None\n    for child in children:\n        if re.fullmatch(r\'h[2-6]\', child.name):\n            current = [child]\n            sections.append(current)\n        elif current is not None:\n            current.append(child)\n    if not sections:\n        blocks = [n for n in children if text(n)]\n        return not (blocks and mostly_links(blocks[-1:]))\n    doomed, verdict = [], True\n    for heading, *body in reversed(sections):\n        title = text(heading)\n        has_text = any(text(n) for n in body)\n        media = any(n.name in {\'img\', \'table\', \'figure\', \'iframe\', \'video\', \'audio\'}\n                    or n.select_one(\'img, table, figure, iframe, video, audio\') for n in body)\n        if WIKI_NAVIGATION_HEADINGS.fullmatch(title) or not has_text and not media:\n            doomed += [heading, *body]\n            continue\n        if title in WIKI_REFERENCE_HEADINGS or not has_text:\n            continue\n        verdict = not mostly_links(body)\n        break\n    removed = {id(node) for node in doomed}\n    if not any(text(n) for n in children if id(n) not in removed):\n        return \'listing\'\n    for node in doomed:\n        remove(node, \'wiki_tail_navigation\')\n    return verdict\n\n\ndef clean_region(root, host, role, result):\n    def remove(node, reason):\n        value = text(node)\n        if value:\n            result[\'removed\'].append({\'reason\': reason, \'selector_path\': dom_path(node), \'text\': value,\n                                      \'links\': [dict(text=text(a), url=urljoin(result[\'url\'], a[\'href\']))\n                                                for a in node.select(\'a[href]\')]})\n        node.decompose()\n\n    common = COMMON_REMOVE.replace(\', article.story\', \'\') if role == \'listing\' else COMMON_REMOVE\n    selectors = common + \', \' + HOST_REMOVE.get(host, \'[data-parser-unused]\')\n    if role.startswith(\'answer_\'):\n        selectors += \', .b_anscont_more\'\n    for node in top_nodes(root.select(selectors)):\n        remove(node, \'identified_ui\')\n    if host.endswith(\'.39.net\'):\n        notices = {\'39健康网(www.39.net)专稿，未经书面授权请勿转载。\', \'未经作者允许授权，禁止转载\',\n                   \'（以上内容仅授权39健康网独家使用，未经版权方授权请勿转载。）\'}\n        notices = {re.sub(r\'\\s+\', \'\', value) for value in notices}\n        candidates = [node for node in root.select(\'p, div, span, strong, b\')\n                      if re.sub(r\'\\s+\', \'\', text(node)) in notices\n                      or re.fullmatch(r\'（(?:(?:实习)?编辑|责任编辑)：[^）]+）\', text(node))]\n        for node in top_nodes(candidates):\n            remove(node, \'publisher_notice_or_editor_metadata\')\n    if host == \'medlatec.vn\':\n        # Styled store cards: keep ordinary clinical links to drugs in the article.\n        candidates = [node for node in root.select(\'div[style]\')\n                      if \'max-width\' in node.get(\'style\', \'\') and node.select_one(\'a[href*="store.medlatec.vn"]\')\n                      and (\'border\' in node.get(\'style\', \'\') or \'background\' in node.get(\'style\', \'\'))]\n        for node in top_nodes(candidates):\n            remove(node, \'store_promotion_card\')\n        for node in list(root.select(\'p\')):\n            if text(node).startswith(\'Để được tư vấn hoặc đặt lịch khám với bác sĩ chuyên khoa\') and node.select_one(\'a[href^="tel:"]\'):\n                remove(node, \'booking_call_to_action\')\n    if host == \'youmed.vn\':\n        for node in list(root.select(\'p\')):\n            if node.parent is None:\n                continue\n            value = text(node)\n            if ((\'tính năng Chat với bác sĩ trên ứng dụng YouMed\' in value and node.select_one(\'a[href]\'))\n                    or (value.startswith(\'Bạn có thể tải ứng dụng YouMed\') and node.select_one(\'a[href]\'))):\n                sibling = node.find_next_sibling()\n                if sibling and sibling.name in {\'ul\', \'ol\'} and all(\n                        text(item).startswith(\'Bước \') for item in sibling.find_all(\'li\', recursive=False)):\n                    remove(sibling, \'app_promotion_steps\')\n                    sibling = node.find_next_sibling()\n                if sibling and sibling.name == \'figure\' and \'ứng dụng YouMed\' in text(sibling):\n                    remove(sibling, \'app_promotion_figure\')\n                remove(node, \'app_promotion_paragraph\')\n    if host in {\'www.a-hospital.com\', \'zhongyibaodian.net\'}:\n        for heading in list(root.select(\'h2, h3\')):\n            if text(heading) not in {\'更多医学百科条目\', \'相关页面\'}:\n                continue\n            sibling = heading.find_next_sibling()\n            if sibling and sibling.name in {\'ul\', \'ol\', \'div\', \'table\'} and sibling.select_one(\'a[href]\'):\n                remove(sibling, \'wiki_related_navigation\')\n                remove(heading, \'wiki_related_navigation_label\')\n        tail = trim_wiki_tail(root, remove)\n        if tail == \'listing\':\n            result[\'page_type\'] = \'listing\'\n            result[\'warnings\'].append(\'listing_not_single_article\')\n        elif not tail:\n            result[\'warnings\'].append(\'wiki_tail_requires_review\')\n    if host == \'qihuangzhishu.com\':\n        for node in list(root.select(\'.cbd > p\')):\n            if text(node).startswith((\'1000余本中医古籍\', \'下载《\', \'下载所有中医书籍\')):\n                remove(node, \'book_download_navigation\')\n    if host == \'baonghean.vn\':\n        for node in list(root.select(\'table.full-img\')):\n            heading = node.find(\'th\')\n            if heading and text(heading) == \'TIN LIÊN QUAN\':\n                remove(node, \'related_articles_table\')\n    # A related-link label is removed only when the remaining text consists\n    # entirely of links. A paragraph mentioning "相关阅读" in prose is retained.\n    label_pattern = r\'(?:[>»\\s]*Xem thêm\\s*:?[\\s:]*|(?:相关阅读(?:推荐)?|相关推荐|编辑推荐|编辑热荐)[：:]?|【相关(?:文章|专题)】)\'\n    for node in list(root.select(\'p, h2, h3, blockquote, .gu_det_left_con_main\')):\n        if node.parent is None:\n            continue\n        if host.endswith(\'.39.net\') and (text(node) in {\'39健康网(www.39.net)专稿，未经书面授权请勿转载。\', \'未经作者允许授权，禁止转载\', \'（以上内容仅授权39健康网独家使用，未经版权方授权请勿转载。）\'} or re.fullmatch(r\'（责任编辑：[^）]+）\', text(node))):\n            remove(node, \'publisher_notice_or_editor_metadata\')\n            continue\n        outside_links = \'\'.join(str(s) for s in node.find_all(string=True) if not s.find_parent(\'a\')).strip()\n        if re.fullmatch(label_pattern, outside_links, re.I):\n            if node.select_one(\'a[href]\'):\n                remove(node, \'related_links_only\')\n            elif re.fullmatch(label_pattern, text(node), re.I):\n                sibling = node.find_next_sibling()\n                while sibling and sibling.name in {\'a\', \'p\', \'ul\', \'ol\'} and (sibling.name == \'a\' or sibling.select_one(\'a[href]\')):\n                    nonlink = \'\' if sibling.name == \'a\' else \'\'.join(str(s) for s in sibling.find_all(string=True) if not s.find_parent(\'a\')).strip()\n                    if re.sub(r\'[\\s:：、，,。.!！?？·—-]\', \'\', nonlink):\n                        break\n                    following = sibling.find_next_sibling()\n                    remove(sibling, \'related_links_only\')\n                    sibling = following\n                remove(node, \'related_links_label\')\n        elif host == \'www.vinmec.com\' and text(node).startswith((\'Để đặt lịch khám tại viện, Quý khách\', \'Quý khách vui lòng bấm số\')) and node.select_one(\'a[href]\'):\n            remove(node, \'booking_call_to_action\')\n        elif host == \'suckhoedoisong.vn\' and text(node) == \'Xem thêm video đang được quan tâm:\':\n            sibling = node.find_next_sibling()\n            if sibling and sibling.get(\'type\') == \'insertembedcode\':\n                remove(sibling, \'recommended_video\')\n            remove(node, \'recommended_video_label\')\n        elif host == \'test.pmphai.com\' and text(node) == \'此内容为收费内容\':\n            heading = node.find_previous_sibling()\n            result[\'missing_sections\'].append(text(heading) if heading else \'\')\n            remove(node, \'paywall_placeholder\')\n    if host == \'ask.39.net\' and role == \'question\':\n        main = root.select_one(\'.txt_ms\')\n        for repeated in list(root.select(\'.txt_bc\')):\n            span = repeated.select_one(\'span\')\n            same_question = main and span and text(main) == text(span)\n            only_repeated_question = main and re.sub(r\'\\s+\', \'\', text(repeated)) == \'补充说明：\' + re.sub(r\'\\s+\', \'\', text(main))\n            if same_question and only_repeated_question:\n                remove(repeated, \'duplicate_question\')\n    for media in root.select(\'audio, video, iframe\'):\n        for string in list(media.find_all(string=True)):\n            if text(string):\n                result[\'removed\'].append({\'reason\': \'browser_media_fallback\', \'selector_path\': dom_path(media), \'text\': text(string)})\n            string.extract()\n    if host.endswith(\'.39.net\'):\n        for node in list(root.select(\'i[title="img"], i[title="timg"]\')):\n            target = text(node)\n            if target.startswith((\'http://\', \'https://\')):\n                result[\'resources\'].append(dict(kind=\'img\', url=target, caption=\'\'))\n                remove(node, \'gallery_image_url_metadata\')\n\ndef parse_page(record, raw):\n    source, encoding, warnings = decode(raw, record.get(\'content_type\', \'\')) if record[\'state\'] == \'html_response\' else (\'\', \'\', [])\n    soup = BeautifulSoup(source, \'html.parser\')\n    host = record[\'host\']\n    result = {\n        \'doc_id\': record[\'doc_id\'], \'url\': record[\'url\'], \'host\': host,\n        \'final_url\': record.get(\'final_url\'), \'raw_path\': record.get(\'raw_path\'),\n        \'source_sha256\': record.get(\'sha256\'),\n        \'parser_version\': VERSION, \'encoding\': encoding, \'title\': title_of(soup, host),\n        \'status\': \'needs_review\', \'page_type\': \'article\', \'warnings\': warnings,\n        \'blocks\': [], \'answers\': [], \'resources\': [], \'removed\': [],\n        \'selected_source_text\': \'\', \'outside_headings\': [], \'outside_paragraphs\': [], \'selectors\': [],\n        \'missing_sections\': [], \'text\': \'\',\n        \'source_links\': [urljoin(record[\'url\'], n[\'href\']) for n in soup.select(\'.article__source a[href]\')],\n        \'author_metadata\': [],\n        \'normalizations\': [], \'pagination_links\': [],\n    }\n    if record[\'state\'] != \'html_response\':\n        result[\'status\'] = record[\'state\']\n        return result\n    decode_cms(soup, result)\n    if not result[\'title\']:\n        warnings.append(\'title_missing\')\n    body_text = text(soup.body or soup)\n    fatal = host == \'www.cnkang.com\' and body_text.startswith(\'Fatal error\') and len(body_text) < 2000\n    soft404 = (urlsplit(record.get(\'final_url\') or record[\'url\']).path == \'/web/404.html\'\n               or result[\'title\'] == \'抱歉，您访问的页面不存在\')\n    main_article = soup.select_one(\'#main-article\') if host == \'www.vinmec.com\' else None\n    missing_data = host == \'www.vinmec.com\' and not soup.select_one(\'.box_quizz_detail\') and (\n        main_article is None or not text(main_article) and not main_article.select_one(\'img,audio,video,iframe\')) and any(\n        text(node).startswith(\'Không tìm thấy dữ liệu bạn yêu cầu\') for node in soup.select(\'#modal-api-error p\'))\n    # Members-only articles answer with the WordPress login page instead of the post.\n    login_wall = host == \'thaythuocvietnam.vn\' and result[\'title\'].startswith(\'Đăng nhập\')\n    if fatal or soft404 or missing_data or login_wall:\n        result.update(status=\'source_error\', page_type=\'error\', selected_source_text=body_text)\n        warnings.append(\'login_required\' if login_wall else \'source_error_in_html\')\n        return result\n    regions = []\n    if host == \'nhathuoclongchau.com.vn\':\n        regions = longchau_regions(soup, result)\n    elif host in {\'www.120ask.com\', \'ask.39.net\'}:\n        result[\'page_type\'] = \'qa\'\n        question_selector = \'#d_msCon\' if host == \'www.120ask.com\' else \'.ask_hid\'\n        result[\'selectors\'] = [question_selector]\n        question = soup.select_one(question_selector)\n        if question:\n            regions.append((question, \'question\'))\n        else:\n            warnings.append(\'question_missing\')\n        answer_selector = \'.b_answerli:not(.zhenshiAD) .b_anscont_cont\' if host == \'www.120ask.com\' else \'.selected .sele_all .sele_txt\'\n        result[\'selectors\'].append(answer_selector)\n        for index, answer in enumerate(top_nodes(soup.select(answer_selector))):\n            regions.append((answer, f\'answer_{index}\'))\n            box = answer.find_parent(class_=\'b_answerli\' if host == \'www.120ask.com\' else \'sele_all\')\n            doctor = box.select_one(\'.doc_name, .b_answertop, .b_doctor\') if box else None\n            timestamp = box.select_one(\'.doc_time, .b_anscont_time\') if box else None\n            date = text(timestamp) if timestamp else \'\'\n            date_match = re.search(r\'\\d{4}-\\d{2}-\\d{2}[\\s\\d:]*\', date)\n            result[\'answers\'].append({\'index\': index, \'doctor_metadata\': text(doctor) if doctor else \'\', \'timestamp\': date_match.group().strip() if date_match else date})\n        if not result[\'answers\']:\n            warnings.append(\'no_answers_in_html\')\n    elif host == \'jbk.39.net\':\n        result[\'page_type\'] = \'encyclopedia\'\n        if soup.select_one(\'.information_box\'):\n            result[\'selectors\'] = [\'.disease\', \'.disease_box:has(.information_box)\']\n            warnings.append(\'overview_only_linked_tabs_not_fetched\')\n        else:\n            result[\'selectors\'] = [\'article.w740 .intro\', \'article.w740 .catalogItem\']\n        regions = [(n, \'body\') for n in top_nodes(soup.select(\', \'.join(result[\'selectors\'])))]\n    else:\n        for selector in RULES.get(host, []):\n            nodes = top_nodes(soup.select(selector))\n            if host == \'www.wujue.com\' and selector == \'#article-main_left #art-leftbox\':\n                nodes = [n for n in nodes if text(n).startswith(\'图片简介\')]\n            if nodes and (any(text(n) for n in nodes) or any(n.select_one(\'img,iframe,audio,video,embed[src],[data-url],[data-vid]\') for n in nodes)):\n                result[\'selectors\'] = [selector]\n                regions = [(n, \'body\') for n in nodes]\n                break\n    if not regions:\n        listing = soup.select_one(\'.timeline, .timeline.content-list\') if host in {\'nhandan.vn\', \'baogialai.com.vn\'} else None\n        candidate_exists = any(soup.select_one(selector) for selector in RULES.get(host, []))\n        summary = next((n for n in soup.select(SUMMARY_SELECTORS) if text(n)), None) if candidate_exists else None\n        # Long prose elsewhere means the body moved (template change), not an empty source body.\n        if summary and any(len(text(p)) >= 80 for p in soup.select(\'p\')\n                           if not p.find_parent([\'header\', \'footer\', \'nav\', \'aside\']) and summary not in p.parents\n                           and p is not summary):\n            summary = None\n        if listing:\n            regions = [(listing, \'listing\')]\n            result[\'selectors\'] = [\'.timeline\']\n            result[\'page_type\'] = \'listing\'\n            warnings.append(\'listing_not_single_article\')\n        elif urlsplit(record[\'url\']).path in {\'\', \'/\'}:\n            result[\'status\'] = \'homepage\'\n            result[\'page_type\'] = \'homepage\'\n            result[\'selected_source_text\'] = text(soup.body or soup)\n            return result\n        elif summary:\n            # The source published only the lead: the article body element is empty.\n            regions = [(summary, \'summary\')]\n            result[\'selectors\'] = [SUMMARY_SELECTORS]\n            warnings.append(\'body_empty_in_source\')\n        else:\n            result[\'status\'] = \'empty_content\' if candidate_exists else \'unknown_template\'\n            if candidate_exists:\n                warnings.append(\'content_region_empty\')\n            result[\'selected_source_text\'] = text(soup.body or soup)\n            return result\n    if result[\'selectors\'] == [\'#booklist\'] or (host == \'dantri.com.vn\' and \'/video/\' in record[\'url\']\n                                               and result[\'selectors\'] == [\'main > .video-highlight, main > .video-grid\']):\n        result[\'page_type\'] = \'listing\'\n        warnings.append(\'listing_not_single_article\')\n    if host == \'www.cnkang.com\' and result[\'selectors\'] == [\'.detail:has(.ysxp02c_table)\']:\n        result[\'page_type\'] = \'recipe\'\n    if host in {\'www.a-hospital.com\', \'zhongyibaodian.net\'} and soup.select_one(\'#filehistory\'):\n        result[\'page_type\'] = \'attachment\'\n        warnings.append(\'attachment_page_requires_review\')\n    result[\'pagination_links\'] = find_pagination(soup, regions, record.get(\'final_url\') or record[\'url\'])\n    if result[\'pagination_links\']:\n        warnings.append(\'pagination_not_fetched\' if any(p[\'same_article\'] for p in result[\'pagination_links\'])\n                        else \'pagination_link_requires_review\')\n    # Keep introductions outside the body, without taking summaries of sidebar cards.\n    if result[\'page_type\'] == \'article\' and host != \'nhathuoclongchau.com.vn\':\n        body_nodes = {id(n) for root, _ in regions for n in [root, *root.descendants]}\n        summaries = top_nodes(soup.select(SUMMARY_SELECTORS + \', \' + SUMMARY_HOST.get(host, \'[data-parser-unused]\')))\n        summaries = [n for n in summaries if id(n) not in body_nodes and not n.find_parent(\'article\', class_=\'story\') and not n.find_parent(class_=\'related-news\')]\n        regions = [(n, \'summary\') for n in summaries] + regions\n    if host == \'tiemchunglongchau.com.vn\' and soup.select_one(\'section[aria-label="Quiz Details Body"]\'):\n        result[\'page_type\'] = \'quiz\'\n        warnings.append(\'quiz_questions_not_in_server_html\')\n    if host == \'www.vinmec.com\' and soup.select_one(\'.box_quizz_detail\'):\n        result[\'page_type\'] = \'quiz\'\n        warnings.append(\'quiz_questions_not_in_server_html\')\n    if host == \'tiemchunglongchau.com.vn\':\n        heading = next((h for h in soup.select(\'h2\') if text(h) == \'NGUỒN THAM KHẢO\'), None)\n        references = heading.find_parent(\'section\') if heading else None\n        if references and not any(references is root or root in references.parents for root, _ in regions):\n            regions.append((references, \'references\'))\n    if host == \'youmed.vn\' and (references := soup.select_one(\'section.article-content details.sources\')):\n        regions.append((references, \'references\'))\n    if host == \'youmed.vn\':\n        article = soup.select_one(\'section.article-content article:not([class])\')\n        result[\'author_metadata\'] = [text(n) for n in article.find_all(recursive=False) if text(n).startswith(\'Tác giả:\')] if article else []\n    if host == \'www.wujue.com\' and (\'/Photo/\' in record[\'url\'] or \'/tuku/\' in record[\'url\']):\n        result[\'page_type\'] = \'gallery\'\n        warnings.append(\'image_content_not_transcribed\')\n    if (host in {\'vov.vn\', \'vov2.vov.vn\'} and \'infographic\' in result[\'title\'].lower()\n            or \'infographic\' in urlsplit(record[\'url\']).path.lower()):\n        result[\'page_type\'] = \'infographic\'\n        warnings.append(\'image_content_not_transcribed\')\n    if host.endswith(\'.39.net\') and soup.select_one(\'[id^="modePhoto"]\'):\n        result[\'page_type\'] = \'gallery\'\n        warnings.append(\'image_content_not_transcribed\')\n    if host == \'thaythuocvietnam.vn\' and \'Câu hỏi số 1\' in result[\'title\']:\n        result[\'page_type\'] = \'placeholder\'\n        warnings.append(\'placeholder_qa\')\n    result[\'selected_source_text\'] = \'\\n\\n\'.join(text(n) for n, _ in regions)\n    selected = {id(n) for root, _ in regions for n in [root, *root.descendants]}\n    result[\'outside_headings\'] = [{\'text\': text(n), \'path\': dom_path(n)} for n in soup.select(\'h1,h2,h3\') if id(n) not in selected and text(n)]\n    result[\'outside_paragraphs\'] = [{\'text\': text(n), \'path\': dom_path(n)} for n in soup.select(\'p, blockquote\') if id(n) not in selected and len(text(n)) >= 80 and not n.find_parent([\'header\', \'footer\', \'nav\', \'form\']) and not n.find([\'p\', \'blockquote\'])]\n    if host == \'nhathuoclongchau.com.vn\':\n        # SSR duplicates of the same JSON fields are already covered.\n        covered = {text(n) for root, _ in regions for n in root.select(\'h1,h2,h3,p,blockquote\')}\n        result[\'outside_headings\'] = [n for n in result[\'outside_headings\'] if n[\'text\'] not in covered]\n        result[\'outside_paragraphs\'] = [n for n in result[\'outside_paragraphs\'] if n[\'text\'] not in covered]\n    if host in {\'baophutho.vn\', \'baothanhhoa.vn\'} and any(re.fullmatch(r\'[A-Za-z0-9+/=\\s]{200,}\', text(n)) for n, _ in regions):\n        result[\'status\'] = \'encoded_content\'\n        warnings.append(\'body_requires_decoder\')\n        return result\n    if host == \'test.pmphai.com\' and \'此内容为收费内容\' in result[\'selected_source_text\']:\n        warnings.append(\'paywalled_sections\')\n    if host == \'www.120ask.com\' and \'图片涉及用户隐私\' in result[\'selected_source_text\']:\n        warnings.append(\'patient_image_unavailable\')\n    if host == \'jb39.com\':\n        warnings.append(\'section_previews_linked_details_not_fetched\')\n    if host == \'www.cnkang.com\' and result[\'selectors\'] == [\'.zh05b\']:\n        result[\'page_type\'] = \'listing\'\n        warnings.append(\'listing_description_only\')\n    if host == \'www.msdmanuals.cn\' and re.search(r\'Chapter_content|Section_content\', result[\'selectors\'][0]):\n        result[\'page_type\'] = \'listing\'\n        warnings.append(\'chapter_index\')\n    for root, role in regions:\n        normalize_markup(root, result)\n        # Freeze source locations before cleanup changes sibling positions.\n        for node in [root, *root.find_all()]:\n            node[\'_source_path\'] = dom_path(node)\n        clean_region(root, host, role, result)\n        pager_urls = {p[\'url\'] for p in result[\'pagination_links\']}\n        pager_nodes = [node for node in root.select(\'a[href]\') if urlsplit(urljoin(\n            record.get(\'final_url\') or record[\'url\'], node[\'href\']))._replace(fragment=\'\').geturl() in pager_urls]\n        for node in pager_nodes:\n            result[\'removed\'].append(dict(reason=\'pagination_control\', selector_path=dom_path(node),\n                                          text=text(node), url=node[\'href\']))\n            node.decompose()\n        # Resources in identified UI have already been removed.\n        for node in root.select(\'img, audio, video, source, iframe, embed[src], a[href], [data-url], [data-src], [data-vid]\'):\n            target = node.get(\'data-url\') or node.get(\'data-vid\') or node.get(\'data-src\') or node.get(\'data-original\') or node.get(\'data-lazy-src\') or node.get(\'src\') or node.get(\'href\')\n            if not target or target.startswith((\'data:\', \'javascript:\')):\n                continue\n            if node.get(\'data-vid\') and not urlsplit(target).scheme and \'/\' in target and \'.\' in target.split(\'/\')[0]:\n                target = \'https://\' + target\n            absolute = urljoin(record.get(\'final_url\') or record[\'url\'], target)\n            embedded_pdf = parse_qs(urlsplit(absolute).query).get(\'url\', [None])[0]\n            if node.name == \'a\' and role != \'references\' and not urlsplit(absolute).path.lower().endswith(\'.pdf\'):\n                continue\n            target_url = embedded_pdf or absolute\n            suffix = urlsplit(target_url).path.lower()\n            kind = (\'pdf\' if suffix.endswith(\'.pdf\') else \'video\' if suffix.endswith((\'.mp4\', \'.webm\', \'.m3u8\'))\n                    else \'audio\' if suffix.endswith((\'.mp3\', \'.wav\', \'.m4a\')) else node.name)\n            if node.name == \'source\' and node.find_parent([\'audio\', \'video\']):\n                kind = node.find_parent([\'audio\', \'video\']).name\n            if node.name not in {\'img\', \'audio\', \'video\', \'source\', \'iframe\', \'a\'} and kind == node.name:\n                kind = \'embedded\'\n            resource = {\'kind\': kind, \'url\': target_url, \'caption\': node.get(\'alt\') or node.get(\'data-title\', \'\')}\n            if resource not in result[\'resources\']:\n                result[\'resources\'].append(resource)\n        blocks = blocks_from(root, role)\n        # Detect accidental text loss in the DOM walker independently of cleanup.\n        expected = re.sub(r\'\\s+\', \'\', root.get_text()).replace(\'\\x00\', \'\')\n        actual = re.sub(r\'\\s+\', \'\', \'\'.join(b[\'text\'] for b in blocks)).replace(\'|\', \'\')\n        if actual != expected.replace(\'|\', \'\'):\n            warnings.append(\'block_text_mismatch\')\n        result[\'blocks\'].extend(blocks)\n    for index, block in enumerate(result[\'blocks\']):\n        block[\'order\'] = index\n    result[\'text\'] = \'\\n\\n\'.join(b[\'text\'] for b in result[\'blocks\'])\n    if \'[links()]\' in result[\'text\']:\n        warnings.append(\'unrendered_publisher_marker\')\n    for answer in result[\'answers\']:\n        answer[\'text\'] = \'\\n\\n\'.join(b[\'text\'] for b in result[\'blocks\'] if b[\'role\'] == f"answer_{answer[\'index\']}")\n    if host == \'thaythuocvietnam.vn\' and \'Câu hỏi số 1\' in result[\'text\'] and \'Câu trả lời từ chuyên gia\' in result[\'text\']:\n        warnings.append(\'placeholder_qa\')\n    if not result[\'text\']:\n        warnings.append(\'media_only\' if result[\'resources\'] else \'empty_extraction\')\n    text_length = sum(len(b[\'text\']) for b in result[\'blocks\'])\n    image_count = sum(r[\'kind\'] == \'img\' for r in result[\'resources\'])\n    if image_count and (text_length < 100 or image_count >= 2 and text_length < 500):\n        warnings.append(\'image_content_not_transcribed\')\n    # A QA page with both a question and an answer is complete even when short.\n    complete_qa = (result[\'page_type\'] == \'qa\' and any(b[\'role\'] == \'question\' for b in result[\'blocks\'])\n                   and any(answer[\'text\'] for answer in result[\'answers\']))\n    if result[\'text\'] and text_length < 200 and not complete_qa:\n        warnings.append(\'short_content_requires_review\')\n    if any(r[\'kind\'] == \'pdf\' for r in result[\'resources\']):\n        warnings.append(\'pdf_not_parsed\')\n    if any(r[\'kind\'] in {\'audio\', \'video\'} for r in result[\'resources\']):\n        warnings.append(\'media_transcript_not_verified\')\n    if any(r[\'kind\'] in {\'iframe\', \'embedded\'} for r in result[\'resources\']):\n        warnings.append(\'embedded_content_not_parsed\')\n    result[\'warnings\'] = list(dict.fromkeys(warnings))\n    result[\'status\'] = \'needs_review\' if warnings else \'extracted_unreviewed\'\n    return result\n\n\ndef write_review(documents, path):\n    # Text only: original website scripts and active HTML are never embedded.\n    data = json.dumps(documents, ensure_ascii=False).replace(\'<\', \'\\\\u003c\').replace(\'>\', \'\\\\u003e\').replace(\'&\', \'\\\\u0026\')\n    page = \'\'\'<!doctype html><meta charset="utf-8"><title>Parser review</title>\n<style>body{font:15px system-ui;margin:24px}input,select{padding:8px}pre{white-space:pre-wrap;overflow-wrap:anywhere}main{display:grid;grid-template-columns:1fr 1fr;gap:24px}section{border:1px solid #ddd;padding:16px}li{margin:8px 0}.warn{color:#a40}</style>\n<h1>Đối chiếu parser pilot</h1><p>Các bản trích chưa được duyệt. Bên trái là vùng nguồn trước khi làm sạch; bên phải là kết quả. Xem heading ngoài vùng chọn để phát hiện nội dung bị bỏ sót.</p>\n<input id="query" placeholder="Host, doc_id, trạng thái" size="45"><select id="docs"></select><div id="info"></div>\n<main><section><h2>Vùng nguồn trước làm sạch</h2><pre id="source"></pre></section><section><h2>Nội dung trích xuất</h2><pre id="output"></pre></section></main>\n<h2>Các vùng đã loại</h2><pre id="removed"></pre><h2>HTML escape đã chuyển thành DOM</h2><pre id="normalizations"></pre><h2>Trang tiếp theo chưa tải</h2><pre id="pagination"></pre><h2>Heading ngoài vùng chọn — cần đối chiếu</h2><pre id="outside"></pre><h2>Đoạn dài ngoài vùng chọn — có thể là nội dung bị sót hoặc gợi ý</h2><pre id="paragraphs"></pre><h2>Ảnh / media / PDF thuộc vùng nội dung</h2><pre id="resources"></pre>\n<script id="data" type="application/json">DATA</script><script>\nconst rows=JSON.parse(document.getElementById(\'data\').textContent), sel=document.getElementById(\'docs\');\nfunction show(){const r=rows[Number(sel.value)];if(!r)return;document.getElementById(\'info\').textContent=`${r.doc_id} | ${r.host} | ${r.status} | ${r.title} | ${r.url} | ${r.selectors.join(\', \')} | ${r.warnings.join(\', \')}`;\nfor(const [id,value] of Object.entries({source:r.selected_source_text,output:(r.blocks||[]).map(b=>`[${b.role}/${b.kind}] ${b.text}`).join(\'\\\\n\\\\n\'),removed:r.removed,normalizations:r.normalizations||[],pagination:r.pagination_links||[],outside:r.outside_headings,paragraphs:r.outside_paragraphs||[],resources:r.resources}))document.getElementById(id).textContent=typeof value===\'string\'?value:JSON.stringify(value,null,2);}\nfunction filter(){const q=document.getElementById(\'query\').value.toLowerCase();sel.replaceChildren();rows.forEach((r,i)=>{if(`${r.doc_id} ${r.host} ${r.status} ${r.warnings.join(\' \')}`.toLowerCase().includes(q)){const o=document.createElement(\'option\');o.value=i;o.textContent=`${r.doc_id} | ${r.host} | ${r.status}`;sel.appendChild(o);}});show();}\nsel.onchange=show;document.getElementById(\'query\').oninput=filter;filter();</script>\'\'\'\n    path.write_text(page.replace(\'DATA\', data), encoding=\'utf-8\')\n\n\ndef run_pilot(root=Path(\'artifacts/host_template_audit\'), output=Path(\'artifacts/parser_review\')):\n    output.mkdir(parents=True, exist_ok=True)\n    documents = []\n    for line in (root / \'pages.jsonl\').read_text().splitlines():\n        record = json.loads(line)\n        raw = gzip.decompress((root / record[\'raw_path\']).read_bytes()) if record.get(\'raw_path\') else b\'\'\n        try:\n            documents.append(parse_page(record, raw))\n        except Exception as error:\n            documents.append({\'doc_id\': record[\'doc_id\'], \'url\': record[\'url\'], \'host\': record[\'host\'], \'title\': \'\', \'status\': \'parser_error\', \'warnings\': [str(error)], \'blocks\': [], \'answers\': [], \'resources\': [], \'removed\': [], \'outside_headings\': [], \'selectors\': [], \'selected_source_text\': \'\'})\n    with (output / \'documents.jsonl\').open(\'w\') as handle:\n        for document in documents:\n            handle.write(json.dumps(document, ensure_ascii=False) + \'\\n\')\n    write_review(documents, output / \'review.html\')\n    print(json.dumps({\'documents\': len(documents), \'statuses\': dict(Counter(d[\'status\'] for d in documents)), \'warnings\': dict(Counter(w for d in documents for w in d[\'warnings\']))}, ensure_ascii=False, indent=2))\n    return documents\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--pilot\', type=Path, default=Path(\'artifacts/host_template_audit\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'artifacts/parser_review\'))\n    args = parser.parse_args()\n    run_pilot(args.pilot, args.output)\n', 'chunking.py': '"""Paragraph-first chunking with exact source offsets.\n\nEvery chunk is text[start:end] of the exported document, so chunk_text is always\na verbatim extract (SUBMISSION.md). Size is counted in words, or in characters/2\nfor Chinese. The scorer divides overlap by the *reference* chunk length, so a\nchunk shorter than 40% of the reference can never match; err on the large side.\n\n    uv run python scripts/chunking.py stats artifacts/corpus_v1/en.parquet --limits 120 250 400\n"""\nimport argparse\nimport re\nfrom dataclasses import dataclass\n\nPARAGRAPH = re.compile(r\'\\n\\s*\\n\')\nSENTENCE_END = re.compile(r\'(?<=[.!?。！？；;])\\s+|(?<=[。！？；])\')\n\n\n@dataclass\nclass Chunk:\n    order: int\n    start: int\n    end: int\n    text: str\n\n\ndef size(text, lang):\n    return len(text) // 2 if lang == \'zh\' else len(text.split())\n\n\ndef spans(text, pattern, start=0, end=None):\n    """Non-empty (start, end) spans of text[start:end] split by pattern, whitespace-trimmed."""\n    end = len(text) if end is None else end\n    cursor = start\n    for match in list(pattern.finditer(text, start, end)) + [None]:\n        stop = match.start() if match else end\n        piece = text[cursor:stop]\n        if piece.strip():\n            left = cursor + len(piece) - len(piece.lstrip())\n            right = cursor + len(piece.rstrip())\n            yield left, right\n        if match:\n            cursor = match.end()\n\n\ndef sentences(text, start, end, lang, limit):\n    """Sentence spans; a sentence above the limit (tables, lists without punctuation)\n    is cut at line breaks, then hard-cut at whitespace (or every 2*limit chars for zh)."""\n    for s_start, s_end in spans(text, SENTENCE_END, start, end):\n        if size(text[s_start:s_end], lang) <= limit:\n            yield s_start, s_end\n            continue\n        for l_start, l_end in spans(text, re.compile(r\'\\n\'), s_start, s_end):\n            if size(text[l_start:l_end], lang) <= limit:\n                yield l_start, l_end\n            elif lang == \'zh\':\n                for cut in range(l_start, l_end, 2 * limit):\n                    yield cut, min(cut + 2 * limit, l_end)\n            else:\n                words = [m.span() for m in re.finditer(r\'\\S+\', text[l_start:l_end])]\n                for i in range(0, len(words), limit):\n                    piece = words[i:i + limit]\n                    yield l_start + piece[0][0], l_start + piece[-1][1]\n\n\ndef units(text, lang, limit):\n    """Paragraphs, with paragraphs above the limit split into sentence groups."""\n    for start, end in spans(text, PARAGRAPH):\n        if size(text[start:end], lang) <= limit:\n            yield start, end\n            continue\n        group = None\n        for s_start, s_end in sentences(text, start, end, lang, limit):\n            if group and size(text[group[0]:s_end], lang) > limit:\n                yield group\n                group = None\n            group = (group[0], s_end) if group else (s_start, s_end)\n        if group:\n            yield group\n\n\ndef is_heading(text, lang):\n    """Short line without closing punctuation: a section title or list label."""\n    text = text.strip()\n    short = len(text) <= 30 if lang == \'zh\' else len(text.split()) <= 12\n    return short and not text.endswith((\'.\', \'!\', \'?\', \':\', \';\', \'。\', \'！\', \'？\', \'：\', \'；\', \'"\', \'”\', \')\'))\n\n\ndef chunk_document(text, lang, limit=350, minimum=40, overlap=50):\n    """Greedily merge adjacent units up to `limit`; fold a tiny tail into the previous chunk.\n\n    Headings never end a chunk: trailing headings move to the next chunk, so a title\n    always stays with the text it introduces. A chunk that continues the same section\n    also repeats the previous chunk\'s last whole sentences (up to `overlap` units);\n    a chunk opening on a heading starts a new section and gets no overlap.\n    """\n    groups = []\n    for unit in units(text, lang, limit):\n        if groups and size(text[groups[-1][0][0]:unit[1]], lang) <= limit:\n            groups[-1].append(unit)\n            continue\n        carried = []\n        while groups and len(groups[-1]) > 1 and is_heading(text[groups[-1][-1][0]:groups[-1][-1][1]], lang):\n            carried.insert(0, groups[-1].pop())\n        groups.append(carried + [unit])\n    merged = [(group[0][0], group[-1][1]) for group in groups]\n    if len(merged) > 1 and size(text[merged[-1][0]:merged[-1][1]], lang) < minimum:\n        tail = merged.pop()\n        merged[-1] = (merged[-1][0], tail[1])\n    chunks = []\n    for order, (start, end) in enumerate(merged):\n        if order and overlap and not is_heading(text[start:end].split(\'\\n\', 1)[0], lang):\n            previous_start, previous_end = merged[order - 1]\n            for s_start, _ in reversed(list(spans(text, SENTENCE_END, previous_start, previous_end))):\n                if size(text[s_start:previous_end], lang) > overlap:\n                    break\n                start = s_start\n        chunks.append(Chunk(order, start, end, text[start:end]))\n    return chunks\n\n\ndef sample_rows(path, sample):\n    """(lang, text) of a random sample, read one row group at a time: take() on the whole\n    text column overflows the 2 GB string offsets of the 1 GB vi corpus."""\n    import numpy as np\n    import pyarrow.parquet as pq\n\n    file = pq.ParquetFile(path)\n    total = file.metadata.num_rows\n    wanted = None\n    if sample and total > sample:\n        wanted = set(np.random.default_rng(0).choice(total, sample, replace=False).tolist())\n    rows, offset = [], 0\n    for group in range(file.metadata.num_row_groups):\n        count = file.metadata.row_group(group).num_rows\n        columns = file.read_row_group(group, columns=[\'lang\', \'text\']).to_pydict()\n        rows += [(columns[\'lang\'][i], columns[\'text\'][i]) for i in range(count)\n                 if wanted is None or offset + i in wanted]\n        offset += count\n    return rows\n\n\ndef stats(path, limits, sample):\n    import numpy as np\n\n    rows = sample_rows(path, sample)\n    langs, texts = [lang for lang, _ in rows], [text for _, text in rows]\n    lengths = np.array([size(t, l) for t, l in zip(texts, langs)])\n    unit = \'chars/2\' if langs and langs[0] == \'zh\' else \'words\'\n    print(f\'{path}: {len(texts):,} documents sampled, length in {unit}\')\n    print(\'  document length p10/p50/p90/p99:\', *np.percentile(lengths, [10, 50, 90, 99]).astype(int))\n    for limit in limits:\n        counts, chunk_sizes = [], []\n        for text, lang in zip(texts, langs):\n            chunks = chunk_document(text, lang, limit)\n            counts.append(len(chunks))\n            chunk_sizes.extend(size(c.text, lang) for c in chunks)\n            assert all(text[c.start:c.end] == c.text for c in chunks)\n        scale = len(texts) if not sample else None\n        print(f\'  limit {limit:>4}: chunks/doc mean {np.mean(counts):.2f}, \'\n              f\'chunk size p50/p90 {int(np.percentile(chunk_sizes, 50))}/{int(np.percentile(chunk_sizes, 90))}, \'\n              f\'single-chunk docs {np.mean(np.array(counts) == 1):.0%}\'\n              + (f\', total {sum(counts):,}\' if scale else \'\'))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\'action\', choices=[\'stats\'])\n    parser.add_argument(\'parquet\')\n    parser.add_argument(\'--limits\', type=int, nargs=\'+\', default=[120, 250, 400])\n    parser.add_argument(\'--sample\', type=int, default=20000, help=\'0 = all documents\')\n    args = parser.parse_args()\n    stats(args.parquet, args.limits, args.sample)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'text_clean.py': '"""Source-text cleaning, applied before chunking. Every chunk row carries CLEAN_VERSION.\n\n1. Line-break tags become line breaks: <br>, <br/>, <br />, &lt;br&gt;, and the fullwidth\n   ＜br＞ / ＜br ／＞ found in about 1,077 zh documents. A run of tags (with the whitespace\n   around it) becomes as many newlines as it had tags or existing newlines, capped at one blank\n   line, so a break next to a paragraph break does not add a second blank line.\n2. zh lead-in: a first paragraph starting with 核心提示 is dropped when the rest of the document\n   repeats its text, because the site shows a summary before the article. A lead-in with no\n   text is dropped too. A standalone summary that does not reappear is kept.\n\n    核心提示：婴儿泪囊炎 婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病……\\n\\n婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病。一是可能……\n    -> 婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病。一是可能……\n\nNothing else changes (no Unicode or whitespace normalisation).\n\n    from text_clean import clean_text\n"""\nimport re\n\nCLEAN_VERSION = \'clean-v1\'\n\nBR_TAG = r\'(?:<br\\s*/?>|＜br\\s*[/／]?\\s*＞|&lt;br\\s*/?&gt;)\'\nBR_RUN = re.compile(rf\'[ \\t\u3000\\n]*(?:{BR_TAG}[ \\t\u3000\\n]*)+\', re.IGNORECASE)\nBR_ONE = re.compile(BR_TAG, re.IGNORECASE)\nLEAD_IN = re.compile(r\'^(?:核心提示[ \\t\u3000]*[:：][ \\t\u3000]*)+\')\nPARAGRAPH_BREAK = re.compile(r\'\\n\\s*\\n\')\n\n\ndef line_break(match):\n    run = match.group(0)\n    return \'\\n\' * min(max(len(BR_ONE.findall(run)), run.count(\'\\n\')), 2)\n\n\ndef drop_lead_in(text):\n    """Drop a top 核心提示 paragraph whose first 40 characters (whitespace removed, trailing\n    ellipsis removed) share a 10-character run with the rest of the document."""\n    if not LEAD_IN.match(text):\n        return text\n    first, *rest = PARAGRAPH_BREAK.split(text, maxsplit=1)\n    rest_text = rest[0] if rest else \'\'\n    body = re.sub(r\'\\s+\', \'\', LEAD_IN.sub(\'\', first)).rstrip(\'….\')\n    if not body:\n        return rest_text\n    head, rest_plain = body[:40], re.sub(r\'\\s+\', \'\', rest_text)\n    if any(head[i:i + 10] in rest_plain for i in range(max(1, len(head) - 9))):\n        return rest_text\n    return text\n\n\ndef clean_text(text, lang):\n    text = BR_RUN.sub(line_break, text)\n    if lang == \'zh\':\n        text = drop_lead_in(text)\n    return text\n', 'dedup_groups.py': '"""Exact-duplicate groups of the Kaggle corpus, without the Pharmacity bad redirects.\n\npharmacity_exclusions(): IDs whose fetched page is another article (the redirect ended on a page\nwithout the original slug), so their text is wrong for their doc_id.\nhash_groups(): one row per text_sha256 among in_corpus documents that are not excluded.\n\ngroup_type: single; url_variant (same host and path, ignoring scheme, www, trailing slash and\nfragment); cross_host (more than one host); same_host_diff_path (redirects to a merged article,\nrepublished abstracts, placeholder pages). Only single and url_variant are fan-out safe: the\nchunks of the representative can be reported for every doc_id of the group.\n\n    uv run python scripts/dedup_groups.py --registry artifacts/kaggle/medxir-corpus/registry.parquet\n"""\nimport argparse\nimport gzip\nimport json\nfrom pathlib import Path\nfrom urllib.parse import unquote, urlsplit\n\nimport pandas as pd\n\nREASON = \'redirected_to_other_article\'\n\n\ndef pharmacity_exclusions(jsonl_gz_path):\n    """doc_id and reason of each record whose final_url path lacks the original slug."""\n    rows = []\n    with gzip.open(jsonl_gz_path, \'rt\', encoding=\'utf-8\') as file:\n        for line in file:\n            record = json.loads(line)\n            slug = unquote(urlsplit(record[\'url\']).path.strip(\'/\'))\n            landed = unquote(urlsplit(record[\'final_url\']).path)\n            if slug not in landed:\n                rows.append((record[\'doc_id\'], REASON))\n    return pd.DataFrame(rows, columns=[\'doc_id\', \'reason\']).sort_values(\'doc_id\').reset_index(drop=True)\n\n\ndef url_key(url):\n    parts = urlsplit(url)\n    return parts.netloc.lower().removeprefix(\'www.\') + parts.path.rstrip(\'/\').lower()\n\n\ndef hash_groups(registry, excluded_ids=()):\n    """One row per text_sha256. rep_doc_id is the smallest doc_id: duplicate texts are identical,\n    and no crawl-vs-archive reason favours one source, so the choice only sets url/title metadata."""\n    rows = registry[registry.state.eq(\'in_corpus\') & ~registry.doc_id.isin(list(excluded_ids))]\n    rows = rows[[\'doc_id\', \'url\', \'host\', \'text_sha256\']].sort_values(\'doc_id\')\n    rows = rows.assign(key=rows.url.map(url_key))\n    grouped = rows.groupby(\'text_sha256\', sort=True)\n    summary = pd.DataFrame({\n        \'rep_doc_id\': grouped.doc_id.min(),\n        \'doc_ids\': grouped.doc_id.agg(lambda ids: [int(i) for i in ids]),\n        \'n_ids\': grouped.size(),\n        \'n_keys\': grouped.key.nunique(),\n        \'n_hosts\': grouped.host.nunique(),\n    }).reset_index()\n    group_type = pd.Series(\'same_host_diff_path\', index=summary.index)\n    group_type = group_type.mask(summary.n_hosts > 1, \'cross_host\')\n    group_type = group_type.mask(summary.n_keys == 1, \'url_variant\')\n    group_type = group_type.mask(summary.n_ids == 1, \'single\')\n    summary[\'group_type\'] = group_type\n    summary[\'fanout_safe\'] = summary.group_type.isin([\'single\', \'url_variant\'])\n    return summary[[\'text_sha256\', \'rep_doc_id\', \'doc_ids\', \'n_ids\', \'group_type\', \'fanout_safe\']]\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\'--registry\', type=Path, default=Path(\'artifacts/kaggle/medxir-corpus/registry.parquet\'))\n    parser.add_argument(\'--pharmacity\', type=Path, default=Path(\'artifacts/recrawl_20261015/pharmacity_v1/documents.jsonl.gz\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'artifacts/aux_v1\'))\n    args = parser.parse_args()\n\n    args.output.mkdir(parents=True, exist_ok=True)\n    exclusions = pharmacity_exclusions(args.pharmacity)\n    exclusions.to_csv(args.output / \'exclusions.csv\', index=False)\n    registry = pd.read_parquet(args.registry)\n    groups = hash_groups(registry, exclusions.doc_id)\n    groups.to_parquet(args.output / \'hash_groups.parquet\', index=False)\n\n    lang_of = registry.set_index(\'doc_id\').lang\n    lang = groups.rep_doc_id.map(lang_of)\n    print(f\'excluded ids: {len(exclusions):,} {exclusions.reason.value_counts().to_dict()}\')\n    print(f\'hash groups: {len(groups):,}; documents covered: {int(groups.n_ids.sum()):,}\')\n    print(\'\\ngroups by type and lang of the representative:\')\n    print(pd.crosstab(groups.group_type, lang, margins=True))\n    rows = groups.assign(lang=lang).groupby([\'group_type\', \'lang\']).n_ids.sum().unstack(fill_value=0)\n    print(\'\\ndocument ids by type and lang (duplicates only, singles included):\')\n    print(rows.assign(total=rows.sum(axis=1)))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'offline_recovery_parser.py': '"""Offline source adapters. No network calls and no changes to the live crawler."""\nimport gzip,hashlib,io,json,re,zlib\nfrom urllib.parse import urlsplit,unquote\nimport zstandard\nimport content_parser as base\nfrom text_clean import clean_text\n\nRECOVERY_VERSION=\'offline-recovery-v1\'\nbase.RULES[\'laodong.vn\']=[\'.art-body\']\nbase.RULES[\'www.familydoctor.com.cn\']=[\'#viewContent\']\nMAX_RAW=8*1024*1024\n\ndef effective_url(final_url, original, host):\n    final_url=final_url or original\n    parts=urlsplit(final_url)\n    if parts.hostname in {\'web.archive.org\',\'www.web.archive.org\'}:\n        match=re.match(r\'^/web/[0-9]{4,14}[a-z_]*/(https?://.+)$\',parts.path)\n        if not match:return original\n        inner=match.group(1)\n        if parts.query:inner+=\'?\'+parts.query\n        if urlsplit(inner).hostname != host:return original\n        return inner\n    return final_url\n\ndef classify(record,raw):\n    sample=raw[:8000].decode(\'utf-8\',errors=\'ignore\').lower()\n    if not raw:return \'empty_response\'\n    if (\'denied by http_custom\' in sample or \'verify you are human\' in sample\n        or \'document.cookie\' in sample and \'location.reload\' in sample\n        or \'captcha\' in sample and len(raw)<10000):return \'access_challenge\'\n    if record.get(\'content_type\') and \'html\' not in record[\'content_type\']:return \'non_html\'\n    if urlsplit(record[\'url\']).path not in (\'\',\'/\') and urlsplit(record[\'final_url\']).path in (\'\',\'/\'):return \'redirected_home\'\n    return \'html_response\'\n\ndef parse_html(job):\n    doc_id,url,host,lang,final_url,ctype,blob,source_path=job\n    result=dict(doc_id=doc_id,url=url,host=host,lang=lang,source_path=source_path,accepted=False)\n    try:\n        if blob[:4]==bytes.fromhex(\'28b52ffd\'):\n            with zstandard.ZstdDecompressor().stream_reader(io.BytesIO(blob)) as reader:raw=reader.read(MAX_RAW+1)\n        elif blob[:2]==bytes.fromhex(\'1f8b\'):\n            with gzip.GzipFile(fileobj=io.BytesIO(blob)) as reader:raw=reader.read(MAX_RAW+1)\n        else:raw=bytes(blob)\n        if len(raw)>MAX_RAW:raise ValueError(\'html_size_limit\')\n        landed=effective_url(final_url,url,host)\n        record=dict(doc_id=doc_id,url=url,host=host,final_url=landed,content_type=(ctype or \'text/html\').lower(),sha256=hashlib.sha256(raw).hexdigest())\n        record[\'state\']=classify(record,raw)\n        document=base.parse_page(record,raw)\n        assert document[\'doc_id\']==doc_id and document[\'url\']==url\n        text=document.get(\'text\') or \'\\n\\n\'.join(b[\'text\'] for b in document.get(\'blocks\',[]) if b.get(\'text\'))\n        warnings=document.get(\'warnings\',[])\n        result.update(status=document[\'status\'],warnings=warnings,text_chars=len(text),title=document.get(\'title\',\'\'),source_final_url=final_url,final_url=landed,source_html_sha256=record[\'sha256\'])\n        critical={\'structured_source_url_mismatch\',\'block_text_mismatch\',\'encoded_content\'}\n        if record[\'state\']!=\'html_response\' or document[\'status\'] not in {\'extracted_unreviewed\',\'needs_review\'} or len(clean_text(text,lang))<30 or critical.intersection(warnings):return result\n        result.update(accepted=True,text=text,text_sha256=hashlib.sha256(text.encode()).hexdigest(),parser_version=base.VERSION+\'+\'+RECOVERY_VERSION,source=\'snute_offline_recovery\')\n    except Exception as exc:result.update(status=\'parser_error\',error_type=type(exc).__name__)\n    return result\n', 'recovery_release_job.py': '"""CPU Kaggle job: parse audited offline HTML and write an isolated additions corpus."""\nfrom pathlib import Path\nfrom collections import Counter\nfrom urllib.request import urlopen\nfrom urllib.parse import quote\nimport gzip\nimport hashlib\nimport json\nimport multiprocessing as mp\nimport shutil\nimport sqlite3\nimport time\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport offline_recovery_parser as worker\nfrom chunking import chunk_document, size\nfrom text_clean import clean_text, CLEAN_VERSION\nfrom dedup_groups import url_key\n\n\nCHUNK_SCHEMA = pa.schema([\n    (\'chunk_id\', pa.string()), (\'text_sha256\', pa.string()), (\'rep_doc_id\', pa.int64()),\n    (\'lang\', pa.string()), (\'order\', pa.int32()), (\'start\', pa.int64()), (\'end\', pa.int64()),\n    (\'chunk_text\', pa.string()), (\'n_units\', pa.int32()), (\'title\', pa.string()),\n    (\'clean_version\', pa.string()), (\'chunk_params\', pa.string()),\n])\n\n\nclass Sink:\n    def __init__(self, path, schema=None):\n        self.path, self.schema, self.writer, self.rows, self.count = path, schema, None, [], 0\n\n    def add(self, row):\n        self.rows.append(row)\n        if len(self.rows) >= 512:\n            self.flush()\n\n    def flush(self):\n        if not self.rows:\n            return\n        table = pa.Table.from_pylist(self.rows, schema=self.schema)\n        if self.writer is None:\n            self.schema = table.schema\n            self.writer = pq.ParquetWriter(self.path, self.schema, compression=\'zstd\')\n        self.writer.write_table(table)\n        self.count += len(self.rows)\n        self.rows.clear()\n\n    def close(self):\n        self.flush()\n        if self.writer:\n            self.writer.close()\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(4 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef run():\n    started = time.monotonic()\n    root = next(Path(\'/kaggle/input\').rglob(\'recovery_manifest.json\')).parent\n    spec = json.loads((root / \'recovery_manifest.json\').read_text())\n    out = Path(\'/kaggle/working/recovery_release_20261029_v2\')\n    out.mkdir(exist_ok=True)\n    scratch = Path(\'/kaggle/working/recovery_scratch\')\n    scratch.mkdir(exist_ok=True)\n    assert sha(root / \'source_matches.parquet\') == spec[\'source_matches_sha256\']\n    matches = pq.read_table(root / \'source_matches.parquet\').to_pylist()\n    expected = {r[\'doc_id\']: r for r in matches}\n    assert len(expected) == len(matches)\n    files = json.loads((root / \'source_files.json\').read_text())\n    paths = {r[\'path\']: r for r in files}\n    by_file = {}\n    for row in matches:\n        by_file.setdefault(row[\'source_path\'], []).append(row)\n\n    base_manifest_path = next(Path(\'/kaggle/input\').rglob(\'chunk_manifest.json\'))\n    baseline = json.loads(base_manifest_path.read_text())\n    assert baseline[\'dataset_version\'] == spec[\'baseline_version\']\n    assert baseline[\'dataset_built_at\'] == spec[\'baseline_built_at\']\n    assert baseline[\'chunk_params\'] == \'350/50/40\' and baseline[\'clean_version\'] == CLEAN_VERSION\n    import inspect\n    params=inspect.signature(chunk_document).parameters\n    assert [params[n].default for n in [\'limit\',\'overlap\',\'minimum\']]==[350,50,40]\n    base_hashes = {}\n    for batch in pq.ParquetFile(base_manifest_path.parent / \'hash_groups.parquet\').iter_batches(\n            batch_size=65536, columns=[\'text_sha256\', \'rep_doc_id\', \'group_type\', \'fanout_safe\']):\n        for r in batch.to_pylist():\n            base_hashes[r[\'text_sha256\']] = (r[\'rep_doc_id\'], r[\'group_type\'], r[\'fanout_safe\'])\n    base_prefixes = {h[:16]: h for h in base_hashes}\n\n    db = sqlite3.connect(scratch / \'parsed.sqlite\')\n    db.execute(\'PRAGMA journal_mode=WAL\')\n    db.execute(\'CREATE TABLE docs(id INTEGER PRIMARY KEY,sha TEXT,url TEXT,host TEXT,lang TEXT,title TEXT,text TEXT,metadata TEXT)\')\n    review = gzip.open(out / \'needs_review.jsonl.gz\', \'wt\', encoding=\'utf-8\')\n    counters, file_audit = Counter(), []\n\n    def save(d):\n        i = d[\'doc_id\']\n        e = expected.get(i)\n        if e is not None:\n            assert e[\'url\'] == d[\'url\'] and e[\'lang\'] == d[\'lang\']\n        if not d.get(\'text\', \'\').strip():\n            return\n        h = hashlib.sha256(d[\'text\'].encode()).hexdigest()\n        metadata = {k: v for k, v in d.items() if k != \'text\'}\n        db.execute(\'INSERT OR IGNORE INTO docs VALUES (?,?,?,?,?,?,?,?)\',\n                   (i, h, d[\'url\'], d[\'host\'], d[\'lang\'], d.get(\'title\', \'\'),\n                    d[\'text\'], json.dumps(metadata, ensure_ascii=False)))\n\n    seed_path = root / \'seed_documents.jsonl.gz\'\n    raw = gzip.decompress(seed_path.read_bytes()) if seed_path.exists() else (root / \'seed_documents.jsonl\').read_bytes()\n    assert hashlib.sha256(raw).hexdigest() == spec[\'seed_raw_sha256\']\n    seeds = [json.loads(line) for line in raw.decode().splitlines()]\n    assert len({d[\'doc_id\'] for d in seeds}) == spec[\'seed_count\']\n    for d in seeds:\n        assert d[\'baseline_state\'] != \'in_corpus\'\n        save(d)\n    db.commit()\n    done = {r[0] for r in db.execute(\'SELECT id FROM docs\')}\n\n    # Four forked workers; blobs are bounded and submitted in groups of 64.\n    # Working disk holds one source DB at a time (largest 14.75 GB) plus outputs.\n    with mp.get_context(\'fork\').Pool(4) as pool:\n        for name in sorted(by_file, key=lambda n: paths[n][\'size\']):\n            entry = paths[name]\n            targets = {r[\'doc_id\']: r for r in by_file[name] if r[\'doc_id\'] not in done}\n            audit = dict(path=name, targets=len(targets))\n            if not targets:\n                audit[\'status\'] = \'covered_by_seed\'; file_audit.append(audit); continue\n            if time.monotonic() - started > 27000:\n                audit[\'status\'] = \'time_budget\'; file_audit.append(audit); break\n            if shutil.disk_usage(scratch).free < entry[\'size\'] + 1024 ** 3:\n                audit[\'status\'] = \'disk_budget\'; file_audit.append(audit); continue\n            local = scratch / \'source.sqlite\'\n            print(f\'Parse {name}: {len(targets):,} IDs; {entry["size"]/1e9:.2f} GB\', flush=True)\n            try:\n                url = (\'https://huggingface.co/datasets/\' + spec[\'source_repository\'] + \'/resolve/\'\n                       + spec[\'source_revision\'] + \'/\' + quote(name, safe=\'/\') + \'?download=true\')\n                for attempt in range(3):\n                    try:\n                        with urlopen(url, timeout=120) as response, local.open(\'wb\') as f:\n                            while True:\n                                block = response.read(4 * 1024 * 1024)\n                                if not block:\n                                    break\n                                f.write(block)\n                        assert local.stat().st_size == entry[\'size\']\n                        if entry.get(\'lfs\'):\n                            assert sha(local) == entry[\'lfs\'][\'oid\']\n                        break\n                    except Exception:\n                        if attempt == 2:\n                            raise\n                        time.sleep(2 ** (attempt + 1))\n                source_db = sqlite3.connect(local.resolve().as_uri() + \'?mode=ro&immutable=1\', uri=True)\n                good, bad = 0, 0\n                try:\n                    source_db.execute(\'PRAGMA query_only=ON\')\n                    source_db.execute(\'PRAGMA cache_size=-32768\')\n                    target_ids = sorted(targets)\n                    for pos in range(0, len(target_ids), 64):\n                        ids = target_ids[pos:pos+64]\n                        sql = \'SELECT id,url,final_url,ctype,html FROM pages WHERE status=200 AND length(html)<2097152 AND id IN (\' + \',\'.join(\'?\' for _ in ids) + \')\'\n                        tasks = []\n                        for i, u, final, ctype, blob in source_db.execute(sql, ids):\n                            e = targets[i]\n                            assert u == e[\'url\']\n                            tasks.append((i, u, e[\'host\'], e[\'lang\'], final, ctype, blob, name))\n                        for d in pool.imap(worker.parse_html, tasks, chunksize=4):\n                            if d[\'accepted\']:\n                                save(d); done.add(d[\'doc_id\']); good += 1\n                            else:\n                                review.write(json.dumps(d, ensure_ascii=False) + \'\\n\'); bad += 1\n                        db.commit()\n                        if pos and pos % 4096 == 0:\n                            print(f\'  {pos:,}/{len(target_ids):,} checked; accepted={good:,}\', flush=True)\n                finally:\n                    source_db.close()\n                audit.update(status=\'parsed\', accepted=good, review=bad)\n            except Exception as exc:\n                audit.update(status=\'error\', error_type=type(exc).__name__, http_code=getattr(exc, \'code\', None))\n                print(\'  source failed:\', audit[\'error_type\'], flush=True)\n            finally:\n                if local.exists(): local.unlink()\n            file_audit.append(audit)\n            (out / \'file_audit.json\').write_text(json.dumps(file_audit, indent=2))\n    (out / \'file_audit.json\').write_text(json.dumps(file_audit, indent=2))\n    review.close()\n    db.execute(\'CREATE INDEX docs_sha ON docs(sha)\'); db.commit()\n\n    doc_sink = Sink(out / \'documents_additions_20261029_v2.parquet\')\n    for i, h, url, host, lang, title, text, metadata in db.execute(\'SELECT * FROM docs ORDER BY id\'):\n        meta = json.loads(metadata)\n        doc_sink.add(dict(doc_id=i, url=url, host=host, lang=lang, title=title, text=text,\n                          text_sha256=h, status=meta.get(\'status\', \'\'), parser_version=meta.get(\'parser_version\', \'\'),\n                          source=meta.get(\'source\', \'\'), warnings_json=json.dumps(meta.get(\'warnings\', []), ensure_ascii=False)))\n    doc_sink.close()\n    chunk_sink = Sink(out / \'chunk_corpus_additions_20261029_v2.parquet\', CHUNK_SCHEMA)\n    group_sink = Sink(out / \'hash_groups_additions_20261029_v2.parquet\')\n    alias_sink = Sink(out / \'aliases_to_existing_20261029_v2.parquet\')\n    todo_sink = Sink(out / \'embedding_todo_20261029_v2.parquet\')\n    new_prefixes, accepted_by_lang = {}, Counter()\n    for h, rep, n in db.execute(\'SELECT sha,min(id),count(*) FROM docs GROUP BY sha ORDER BY sha\'):\n        members = db.execute(\'SELECT id,url,host FROM docs WHERE sha=? ORDER BY id\', (h,)).fetchall()\n        if h in base_hashes:\n            old_rep, old_type, old_safe = base_hashes[h]\n            for i, url, host in members:\n                alias_sink.add(dict(doc_id=i, url=url, text_sha256=h, existing_rep_doc_id=old_rep,\n                                    existing_group_type=old_type, existing_fanout_safe=bool(old_safe),\n                                    decision=\'mapping_review_required\'))\n            continue\n        assert h[:16] not in base_prefixes or base_prefixes[h[:16]] == h, \'chunk prefix collision with baseline\'\n        assert h[:16] not in new_prefixes or new_prefixes[h[:16]] == h, \'new chunk prefix collision\'\n        new_prefixes[h[:16]] = h\n        lang, title, text = db.execute(\'SELECT lang,title,text FROM docs WHERE id=?\', (rep,)).fetchone()\n        keys = {url_key(url) for _, url, _ in members}; hosts = {host for _, _, host in members}\n        typ = \'single\' if n == 1 else \'url_variant\' if len(keys) == 1 else \'cross_host\' if len(hosts) > 1 else \'same_host_diff_path\'\n        group_sink.add(dict(text_sha256=h, rep_doc_id=rep, doc_ids=[i for i, _, _ in members], n_ids=n,\n                            group_type=typ, fanout_safe=typ in {\'single\', \'url_variant\'}))\n        cleaned = clean_text(text, lang)\n        for c in chunk_document(cleaned, lang):\n            assert cleaned[c.start:c.end] == c.text\n            cid = f\'{h[:16]}-{c.order}\'\n            chunk_sink.add(dict(chunk_id=cid, text_sha256=h, rep_doc_id=rep, lang=lang, order=c.order,\n                                start=c.start, end=c.end, chunk_text=c.text, n_units=size(c.text, lang),\n                                title=title, clean_version=CLEAN_VERSION, chunk_params=\'350/50/40\'))\n            todo_sink.add(dict(chunk_id=cid, text_sha256=h, chunk_text_sha256=hashlib.sha256(c.text.encode()).hexdigest(),\n                               embedding_state=\'check_each_model_inventory_before_embedding\'))\n            accepted_by_lang[lang] += 1\n    for sink in [chunk_sink, group_sink, alias_sink, todo_sink]: sink.close()\n    original_vi = json.loads((root / \'all_vi_new_records.json\').read_text())\n    wanted_vi = {r[\'doc_id\'] for r in original_vi if r[\'usable_text\']}\n    included = {r[0] for r in db.execute(\'SELECT id FROM docs\')}\n    missing_vi = sorted(wanted_vi - included)\n    (out / \'vi_crawl_ids_not_in_release.json\').write_text(json.dumps(missing_vi))\n    report = dict(release=spec[\'release\'], baseline_manifest=baseline, accepted_document_ids=doc_sink.count,\n                  source_candidate_ids=len(matches), seed_ids=len(seeds), new_text_groups=group_sink.count,\n                  new_chunks=chunk_sink.count, alias_document_ids=alias_sink.count, chunks_by_language=dict(accepted_by_lang),\n                  vi_completed_ids_not_in_baseline=spec[\'all_vi_not_in_v1\'], vi_with_text_expected=len(wanted_vi),\n                  vi_with_text_included=len(wanted_vi & included), vi_with_text_missing=len(missing_vi),\n                  source_file_statuses=dict(Counter(r[\'status\'] for r in file_audit)),\n                  baseline_modified=False, embeddings_generated=False, elapsed_seconds=round(time.monotonic()-started, 1),\n                  replaces_previous_addition_files=True,\n                  limitations=[\'Parsed candidates may retain needs_review warnings; inspect review output.\',\n                               \'Aliases to existing text require mapping review; no automatic fanout.\',\n                               \'Compare chunk inventory separately for every model/revision before embedding.\'])\n    db.close()\n    for p in scratch.glob(\'parsed.sqlite*\'): p.unlink()\n    report[\'files\'] = {p.name: dict(bytes=p.stat().st_size, sha256=sha(p)) for p in out.iterdir()\n                       if p.is_file() and p.name != \'release_manifest.json\'}\n    (out / \'release_manifest.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2))\n    print(json.dumps({k: v for k, v in report.items() if k not in [\'files\', \'baseline_manifest\']}, indent=2))\n\n\nif __name__ == \'__main__\':\n    run()\n'}
from pathlib import Path
import sys,subprocess,time
p=Path('/kaggle/working/recovery_modules');p.mkdir(exist_ok=True)
for name,text in MODULES.items():(p/name).write_text(text,encoding='utf-8')
RUNNER_LOG=Path('/kaggle/working/recovery_runner.log')
LOG_HANDLE=RUNNER_LOG.open('w')
PROCESS=subprocess.Popen([sys.executable,'-u',str(p/'recovery_release_job.py')],stdout=LOG_HANDLE,stderr=subprocess.STDOUT)
print('CPU recovery pid:',PROCESS.pid)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
deadline=time.monotonic()+590
while PROCESS.poll() is None and time.monotonic()<deadline:
    time.sleep(20)
    if RUNNER_LOG.exists():
        with RUNNER_LOG.open('rb') as f:
            f.seek(max(0,RUNNER_LOG.stat().st_size-1500))
            lines=f.read().decode('utf-8',errors='replace').splitlines()
        print('\n'.join(lines[-3:]),flush=True)
if PROCESS.poll() is not None:print('CPU recovery exit:',PROCESS.returncode)


In [ ]:
if PROCESS.poll() is None:
    PROCESS.terminate()
    raise TimeoutError('CPU recovery exceeded its eight-hour notebook budget')
LOG_HANDLE.close()
assert PROCESS.returncode==0,RUNNER_LOG.read_text(errors='replace')[-5000:]
summary=Path('/kaggle/working/recovery_release_20261029_v2/release_manifest.json')
assert summary.is_file()
print(summary.read_text()[:5000])
